In [0]:


from pyspark.sql import functions as F

# Dataset sizes
NUM_CUSTOMERS = 10000
NUM_PRODUCTS = 500
NUM_EVENTS = 100000

# -----------------------------------
# 1. GENERATE CUSTOMERS
# -----------------------------------

customers = (
    spark.range(NUM_CUSTOMERS)
    .withColumn(
        "Customer_ID",
        F.format_string("C%05d", F.col("id") + 1)
    )
    .withColumn("Age", (F.floor(F.rand(1) * 43) + 18).cast("int"))
    .withColumn(
        "Gender",
        F.when(F.rand(2) < 0.5, "Male").otherwise("Female")
    )
    .withColumn(
        "City",
        F.element_at(
            F.array(
                F.lit("Bengaluru"),
                F.lit("Mumbai"),
                F.lit("Delhi"),
                F.lit("Hyderabad"),
                F.lit("Pune"),
                F.lit("Chennai"),
                F.lit("Kolkata"),
                F.lit("Jaipur")
            ),
            (F.floor(F.rand(3) * 8) + 1).cast("int")
        )
    )
    .withColumn(
        "Signup_Date",
        F.date_sub(
            F.current_date(),
            (F.floor(F.rand(4) * 730)).cast("int")
        )
    )
    .drop("id")
)

# -----------------------------------
# 2. GENERATE PRODUCTS
# -----------------------------------

products = (
    spark.range(NUM_PRODUCTS)
    .withColumn(
        "Product_ID",
        F.format_string("P%04d", F.col("id") + 1)
    )
    .withColumn(
        "Category",
        F.element_at(
            F.array(
                F.lit("Fashion"),
                F.lit("Footwear"),
                F.lit("Electronics"),
                F.lit("Home"),
                F.lit("Beauty"),
                F.lit("Sports")
            ),
            (F.floor(F.rand(5) * 6) + 1).cast("int")
        )
    )
    .withColumn(
        "Brand",
        F.concat(F.lit("Brand "), F.element_at(
            F.array(*[F.lit(x) for x in ["A", "B", "C", "D", "E", "F", "G", "H"]]),
            (F.floor(F.rand(6) * 8) + 1).cast("int")
        ))
    )
    .withColumn(
        "Subcategory",
        F.element_at(
            F.array(
                F.lit("Shirts"), F.lit("Shoes"),
                F.lit("Audio"), F.lit("Accessories"),
                F.lit("Home Decor"), F.lit("Skincare"),
                F.lit("Fitness"), F.lit("Wearables")
            ),
            (F.floor(F.rand(7) * 8) + 1).cast("int")
        )
    )
    .withColumn(
        "Product_Name",
        F.concat(
            F.col("Category"),
            F.lit(" "),
            F.col("Subcategory"),
            F.lit(" "),
            F.col("Product_ID")
        )
    )
    .withColumn(
        "Price",
        (F.floor(F.rand(8) * 14700) + 299).cast("int")
    )
    .withColumn(
        "Colour",
        F.element_at(
            F.array(
                F.lit("Black"), F.lit("White"),
                F.lit("Blue"), F.lit("Grey"),
                F.lit("Red"), F.lit("Green"),
                F.lit("Silver")
            ),
            (F.floor(F.rand(9) * 7) + 1).cast("int")
        )
    )
    .withColumn(
        "Material",
        F.element_at(
            F.array(
                F.lit("Cotton"), F.lit("Polyester"),
                F.lit("Leather"), F.lit("Metal"),
                F.lit("Plastic"), F.lit("Linen")
            ),
            (F.floor(F.rand(10) * 6) + 1).cast("int")
        )
    )
    .withColumn(
        "Fit",
        F.element_at(
            F.array(
                F.lit("Regular"), F.lit("Slim"),
                F.lit("Relaxed"), F.lit("Oversized")
            ),
            (F.floor(F.rand(11) * 4) + 1).cast("int")
        )
    )
    .withColumn(
        "Rating",
        F.round(F.rand(12) * 1.5 + 3.5, 1)
    )
    .drop("id")
    .select(
        "Product_ID", "Product_Name", "Category",
        "Subcategory", "Brand", "Price", "Colour",
        "Material", "Fit", "Rating"
    )
)

# -----------------------------------
# 3. GENERATE CUSTOMER ACTIVITY EVENTS
# -----------------------------------

events = (
    spark.range(NUM_EVENTS)
    .withColumn(
        "Event_ID",
        F.format_string("E%07d", F.col("id") + 1)
    )
    .withColumn(
        "Customer_ID",
        F.format_string(
            "C%05d",
            (F.floor(F.rand(13) * NUM_CUSTOMERS) + 1).cast("int")
        )
    )
    .withColumn(
        "Product_ID",
        F.format_string(
            "P%04d",
            (F.floor(F.rand(14) * NUM_PRODUCTS) + 1).cast("int")
        )
    )
    .withColumn(
        "Action",
        F.when(F.rand(15) < 0.65, "View")
         .when(F.rand(15) < 0.90, "Cart")
         .otherwise("Purchase")
    )
    .withColumn(
        "Event_Timestamp",
        F.from_unixtime(
            F.unix_timestamp(F.current_timestamp()) -
            (F.rand(16) * 90 * 86400).cast("int")
        ).cast("timestamp")
    )
    .drop("id")
)

# -----------------------------------
# 4. SAVE AS SEPARATE DELTA TABLES
# -----------------------------------

customers.write.format("delta").mode("overwrite").saveAsTable(
    "default.thames_customers_large_raw"
)

products.write.format("delta").mode("overwrite").saveAsTable(
    "default.thames_products_large_raw"
)

events.write.format("delta").mode("overwrite").saveAsTable(
    "default.thames_events_large_raw"
)

print("THAMES large datasets generated successfully!")


THAMES large datasets generated successfully!


In [0]:
for table in [
    "thames_customers_large_raw",
    "thames_products_large_raw",
    "thames_events_large_raw"
]:
    df = spark.table(f"default.{table}")
    print(f"{table}: {df.count()} records")
    df.show(5, truncate=False)

thames_customers_large_raw: 10000 records
+-----------+---+------+---------+-----------+
|Customer_ID|Age|Gender|City     |Signup_Date|
+-----------+---+------+---------+-----------+
|C00001     |18 |Male  |Mumbai   |2025-01-01 |
|C00002     |56 |Male  |Bengaluru|2026-02-02 |
|C00003     |24 |Male  |Pune     |2026-02-12 |
|C00004     |49 |Male  |Chennai  |2026-06-07 |
|C00005     |32 |Male  |Chennai  |2024-12-17 |
+-----------+---+------+---------+-----------+
only showing top 5 rows
thames_products_large_raw: 500 records
+----------+------------------------+--------+-----------+-------+-----+------+---------+---------+------+
|Product_ID|Product_Name            |Category|Subcategory|Brand  |Price|Colour|Material |Fit      |Rating|
+----------+------------------------+--------+-----------+-------+-----+------+---------+---------+------+
|P0313     |Sports Accessories P0313|Sports  |Accessories|Brand G|1185 |Black |Plastic  |Oversized|4.6   |
|P0314     |Fashion Audio P0314     |Fashion

In [0]:

from pyspark.sql import functions as F

# Load the new raw Delta tables
customers = spark.table("default.thames_customers_large_raw")
events = spark.table("default.thames_events_large_raw")
products = spark.table("default.thames_products_large_raw")

# Join activity events with product prices
activity = events.join(
    products.select("Product_ID", "Price"),
    on="Product_ID",
    how="left"
)

# Aggregate customer behaviour
customer_activity = (
    activity
    .groupBy("Customer_ID")
    .agg(
        F.sum(F.when(F.col("Action") == "View", 1).otherwise(0))
         .alias("Total_Views"),

        F.sum(F.when(F.col("Action") == "Cart", 1).otherwise(0))
         .alias("Total_Carts"),

        F.sum(F.when(F.col("Action") == "Purchase", 1).otherwise(0))
         .alias("Total_Purchases"),

        F.sum(
            F.when(F.col("Action") == "Purchase", F.col("Price"))
             .otherwise(0)
        ).alias("Total_Spending"),

        F.count("*").alias("Total_Interactions"),

        F.max("Event_Timestamp").alias("Last_Activity")
    )
)

# Include all customers, even those without activity
customer_features = (
    customers
    .join(customer_activity, on="Customer_ID", how="left")
    .fillna({
        "Total_Views": 0,
        "Total_Carts": 0,
        "Total_Purchases": 0,
        "Total_Spending": 0,
        "Total_Interactions": 0
    })
    .withColumn(
        "Average_Order_Value",
        F.when(
            F.col("Total_Purchases") > 0,
            F.round(
                F.col("Total_Spending") / F.col("Total_Purchases"), 2
            )
        ).otherwise(0)
    )
    .withColumn(
        "Days_Since_Last_Activity",
        F.datediff(
            F.current_date(),
            F.to_date("Last_Activity")
        )
    )
)

# Save as a new Delta table
customer_features.write.format("delta").mode("overwrite").saveAsTable(
    "default.thames_customer_features_large"
)

print("Customer features generated successfully!")

customer_features.show(10, truncate=False)


Customer features generated successfully!
+-----------+---+------+---------+-----------+-----------+-----------+---------------+--------------+------------------+-------------------+-------------------+------------------------+
|Customer_ID|Age|Gender|City     |Signup_Date|Total_Views|Total_Carts|Total_Purchases|Total_Spending|Total_Interactions|Last_Activity      |Average_Order_Value|Days_Since_Last_Activity|
+-----------+---+------+---------+-----------+-----------+-----------+---------------+--------------+------------------+-------------------+-------------------+------------------------+
|C00001     |18 |Male  |Mumbai   |2025-01-01 |9          |3          |0              |0             |12                |2026-10-27 20:20:12|0.0                |9                       |
|C00002     |56 |Male  |Bengaluru|2026-02-02 |6          |3          |0              |0             |9                 |2026-11-05 14:38:17|0.0                |0                       |
|C00003     |24 |Male  |Pune

In [0]:
large_features = spark.table(
    "default.thames_customer_features_large"
)

print("Total customers:", large_features.count())

large_features.select(
    F.sum("Total_Views").alias("Views"),
    F.sum("Total_Carts").alias("Carts"),
    F.sum("Total_Purchases").alias("Purchases"),
    F.sum("Total_Spending").alias("Revenue")
).show()

large_features.describe(
    "Total_Views",
    "Total_Carts",
    "Total_Purchases",
    "Total_Spending",
    "Average_Order_Value"
).show()

Total customers: 10000
+-----+-----+---------+--------+
|Views|Carts|Purchases| Revenue|
+-----+-----+---------+--------+
|64896|31533|     3571|26469554|
+-----+-----+---------+--------+

+-------+-----------------+------------------+------------------+-----------------+-------------------+
|summary|      Total_Views|       Total_Carts|   Total_Purchases|   Total_Spending|Average_Order_Value|
+-------+-----------------+------------------+------------------+-----------------+-------------------+
|  count|            10000|             10000|             10000|            10000|              10000|
|   mean|           6.4896|            3.1533|            0.3571|        2646.9554| 2240.6115999999993|
| stddev|2.570516017312384|1.7800887566630386|0.5948234792752798|5053.603356764362|  4054.612400234258|
|    min|                0|                 0|                 0|                0|                0.0|
|    max|               20|                11|                 4|            38400|

In [0]:

from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Load customer features
features = spark.table("default.thames_customer_features_large")

# Handle customers with no activity
features = features.fillna({
    "Total_Views": 0,
    "Total_Carts": 0,
    "Total_Purchases": 0,
    "Total_Spending": 0,
    "Average_Order_Value": 0,
    "Days_Since_Last_Activity": 91
})

# Select behavioural features
input_columns = [
    "Total_Views",
    "Total_Carts",
    "Total_Purchases",
    "Total_Spending",
    "Days_Since_Last_Activity"
]

# Assemble features
assembler = VectorAssembler(
    inputCols=input_columns,
    outputCol="raw_features"
)

assembled = assembler.transform(features)

# Standardize features
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="scaled_features",
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(assembled)
scaled_data = scaler_model.transform(assembled)

# Compare different K values
evaluator = ClusteringEvaluator(
    featuresCol="scaled_features",
    predictionCol="prediction",
    metricName="silhouette"
)

results = []

for k in [2, 3, 4, 5]:
    kmeans = KMeans(
        k=k,
        seed=42,
        featuresCol="scaled_features",
        predictionCol="prediction"
    )

    model = kmeans.fit(scaled_data)
    predictions = model.transform(scaled_data)

    score = evaluator.evaluate(predictions)

    results.append((k, score))

    print(f"K = {k}, Silhouette Score = {score:.4f}")

# Select the best K
best_k, best_score = max(results, key=lambda x: x[1])

print(f"\nBest K: {best_k}")
print(f"Best Silhouette Score: {best_score:.4f}")

# Train final model using best K
final_kmeans = KMeans(
    k=best_k,
    seed=42,
    featuresCol="scaled_features",
    predictionCol="Cluster"
)

final_model = final_kmeans.fit(scaled_data)
final_predictions = final_model.transform(scaled_data)

# Save customer segmentation
customer_segments_large = final_predictions.select(
    "Customer_ID",
    "Age",
    "Gender",
    "City",
    "Total_Views",
    "Total_Carts",
    "Total_Purchases",
    "Total_Spending",
    "Average_Order_Value",
    "Days_Since_Last_Activity",
    F.col("Cluster").cast("int")
)

customer_segments_large.write.format("delta").mode("overwrite").saveAsTable(
    "default.thames_customer_segments_large"
)

print("\nLarge customer segmentation saved successfully!")


K = 2, Silhouette Score = 0.5139
K = 3, Silhouette Score = 0.4469
K = 4, Silhouette Score = 0.3648
K = 5, Silhouette Score = 0.3690

Best K: 2
Best Silhouette Score: 0.5139

Large customer segmentation saved successfully!


In [0]:
segments = spark.table("default.thames_customer_segments_large")

# Customer distribution across clusters
segments.groupBy("Cluster").count().orderBy("Cluster").show()

# Average customer behaviour per cluster
segments.groupBy("Cluster").agg(
    F.count("*").alias("Customers"),
    F.round(F.avg("Total_Views"), 2).alias("Avg_Views"),
    F.round(F.avg("Total_Carts"), 2).alias("Avg_Carts"),
    F.round(F.avg("Total_Purchases"), 2).alias("Avg_Purchases"),
    F.round(F.avg("Total_Spending"), 2).alias("Avg_Spending"),
    F.round(F.avg("Days_Since_Last_Activity"), 2).alias("Avg_Days_Inactive")
).orderBy("Cluster").show()

+-------+-----+
|Cluster|count|
+-------+-----+
|      0| 2994|
|      1| 7006|
+-------+-----+

+-------+---------+---------+---------+-------------+------------+-----------------+
|Cluster|Customers|Avg_Views|Avg_Carts|Avg_Purchases|Avg_Spending|Avg_Days_Inactive|
+-------+---------+---------+---------+-------------+------------+-----------------+
|      0|     2994|     6.49|     3.13|         1.18|     8835.62|             8.17|
|      1|     7006|     6.49|     3.16|          0.0|        2.24|             9.26|
+-------+---------+---------+---------+-------------+------------+-----------------+



In [0]:
events = spark.table("default.thames_events_large_raw")

events.groupBy("Action").count().show()

events.groupBy("Action").agg(
    F.round(
        F.count("*") * 100.0 / events.count(), 2
    ).alias("Percentage")
).show()

+--------+-----+
|  Action|count|
+--------+-----+
|    View|64896|
|    Cart|31533|
|Purchase| 3571|
+--------+-----+

+--------+----------+
|  Action|Percentage|
+--------+----------+
|    View|      64.9|
|    Cart|     31.53|
|Purchase|      3.57|
+--------+----------+



In [0]:

from pyspark.sql import functions as F

NUM_EVENTS = 100000
NUM_CUSTOMERS = 10000
NUM_PRODUCTS = 500

events = (
    spark.range(NUM_EVENTS)
    .withColumn(
        "Event_ID",
        F.format_string("E%07d", F.col("id") + 1)
    )
    .withColumn(
        "Customer_ID",
        F.format_string(
            "C%05d",
            (F.floor(F.rand(13) * NUM_CUSTOMERS) + 1).cast("int")
        )
    )
    .withColumn(
        "Product_ID",
        F.format_string(
            "P%04d",
            (F.floor(F.rand(14) * NUM_PRODUCTS) + 1).cast("int")
        )
    )
    .withColumn("action_random", F.rand(15))
    .withColumn(
        "Action",
        F.when(F.col("action_random") < 0.65, "View")
         .when(F.col("action_random") < 0.90, "Cart")
         .otherwise("Purchase")
    )
    .withColumn(
        "Event_Timestamp",
        F.from_unixtime(
            F.unix_timestamp(F.current_timestamp()) -
            (F.rand(16) * 90 * 86400).cast("int")
        ).cast("timestamp")
    )
    .drop("id", "action_random")
)

events.write.format("delta").mode("overwrite").saveAsTable(
    "default.thames_events_large_raw"
)

print("Activity events regenerated successfully!")


Activity events regenerated successfully!


In [0]:
spark.table("default.thames_events_large_raw") \
    .groupBy("Action").count().show()
    

+--------+-----+
|  Action|count|
+--------+-----+
|    Cart|25042|
|    View|64896|
|Purchase|10062|
+--------+-----+



In [0]:
from pyspark.ml.feature import StringIndexer

# Convert customer IDs into numeric user IDs
customer_indexer = StringIndexer(
    inputCol="Customer_ID",
    outputCol="user_id",
    handleInvalid="skip"
)

customer_indexer_model = customer_indexer.fit(
    events.select("Customer_ID").distinct()
)

customer_mapping = customer_indexer_model.transform(
    events.select("Customer_ID").distinct()
).select(
    "Customer_ID",
    F.col("user_id").cast("int")
)

# Convert product IDs into numeric product IDs
product_indexer = StringIndexer(
    inputCol="Product_ID",
    outputCol="product_id",
    handleInvalid="skip"
)

product_indexer_model = product_indexer.fit(
    products.select("Product_ID").distinct()
)

product_mapping = product_indexer_model.transform(
    products.select("Product_ID").distinct()
).select(
    "Product_ID",
    F.col("product_id").cast("int")
)

# Create ALS training data
training_data = (
    interactions
    .join(customer_mapping, "Customer_ID")
    .join(product_mapping, "Product_ID")
    .select("user_id", "product_id", "Rating")
)

In [0]:
recommendations = spark.table(
    "default.thames_product_recommendations_large"
)

print("Total recommendations:", recommendations.count())

print(
    "Unique customers:",
    recommendations.select("Customer_ID").distinct().count()
)

recommendations.orderBy(
    "Customer_ID", "Recommendation_Rank"
).show(20, truncate=False)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6720128815377622>, line 5
      1 recommendations = spark.table(
      2     "default.thames_product_recommendations_large"
      3 )
----> 5 print("Total recommendations:", recommendations.count())
      7 print(
      8     "Unique customers:",
      9     recommendations.select("Customer_ID").distinct().count()
     10 )
     12 recommendations.orderBy(
     13     "Customer_ID", "Recommendation_Rank"
     14 ).show(20, truncate=False)

File <command-6720128815377622>, line 1
----> 1 recommendations = spark.table(
      2     "default.thames_product_recommendations_large"
      3 )
      5 print("Total recommendations:", recommendations.count())
      7 print(
      8     "Unique customers:",
      9     recommendations.select("Customer_ID").distinct().count()
     10 )

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The 

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import StringIndexer
from pyspark.ml.recommendation import ALS

# ==========================================
# 1. LOAD CLEAN SOURCE TABLES
# ==========================================

events = spark.table(
    "default.thames_events_large_raw"
).select(
    "Customer_ID", "Product_ID", "Action"
)

products = spark.table(
    "default.thames_products_large_raw"
).select(
    "Product_ID", "Product_Name", "Category",
    "Price", "Rating"
)

# ==========================================
# 2. CREATE CUSTOMER-PRODUCT INTERACTIONS
# ==========================================

interactions = (
    events
    .withColumn(
        "Weight",
        F.when(F.col("Action") == "View", 1.0)
         .when(F.col("Action") == "Cart", 3.0)
         .when(F.col("Action") == "Purchase", 5.0)
    )
    .groupBy("Customer_ID", "Product_ID")
    .agg(F.sum("Weight").alias("Interaction_Rating"))
)

# ==========================================
# 3. CREATE NUMERIC ID MAPPINGS
# ==========================================

customer_ids = events.select("Customer_ID").distinct()
product_ids = products.select("Product_ID").distinct()

customer_indexer = StringIndexer(
    inputCol="Customer_ID",
    outputCol="user_id"
)

customer_indexer_model = customer_indexer.fit(customer_ids)

customer_mapping = (
    customer_indexer_model.transform(customer_ids)
    .select(
        F.col("Customer_ID").alias("customer_key"),
        F.col("user_id").cast("int").alias("user_key")
    )
)

product_indexer = StringIndexer(
    inputCol="Product_ID",
    outputCol="item_id"
)

product_indexer_model = product_indexer.fit(product_ids)

product_mapping = (
    product_indexer_model.transform(product_ids)
    .select(
        F.col("Product_ID").alias("product_key"),
        F.col("item_id").cast("int").alias("item_key")
    )
)

# ==========================================
# 4. PREPARE ALS TRAINING DATA
# ==========================================

training_data = (
    interactions
    .join(
        customer_mapping,
        interactions.Customer_ID == customer_mapping.customer_key,
        "inner"
    )
    .join(
        product_mapping,
        interactions.Product_ID == product_mapping.product_key,
        "inner"
    )
    .select(
        F.col("user_key").alias("user_id"),
        F.col("item_key").alias("product_id"),
        F.col("Interaction_Rating").alias("rating")
    )
)

print("Training interactions:", training_data.count())

# ==========================================
# 5. TRAIN ALS MODEL
# ==========================================

als = ALS(
    userCol="user_id",
    itemCol="product_id",
    ratingCol="rating",
    rank=10,
    maxIter=10,
    regParam=0.1,
    implicitPrefs=True,
    alpha=20.0,
    coldStartStrategy="drop",
    seed=42
)

als_model = als.fit(training_data)

print("ALS model trained successfully!")


# ==========================================
# 6. GENERATE CUSTOMER-PRODUCT CANDIDATES
# ==========================================

# All possible customer and product combinations
customer_numeric_ids = customer_mapping.select(
    F.col("user_key").alias("user_id")
)

product_numeric_ids = product_mapping.select(
    F.col("item_key").alias("product_id")
)

candidate_pairs = customer_numeric_ids.crossJoin(
    product_numeric_ids
)

# Remove combinations already present in interaction history
seen_numeric = training_data.select(
    "user_id", "product_id"
).distinct()

unseen_pairs = candidate_pairs.join(
    seen_numeric,
    ["user_id", "product_id"],
    "left_anti"
)

print("Unseen customer-product combinations:", unseen_pairs.count())

# ==========================================
# 7. SCORE CANDIDATES USING TRAINED ALS MODEL
# ==========================================

scored = als_model.transform(unseen_pairs)

# Remove invalid predictions
scored = scored.filter(
    F.col("prediction").isNotNull()
)

# ==========================================
# 8. SELECT TOP 5 PRODUCTS PER CUSTOMER
# ==========================================

ranking = Window.partitionBy("user_id").orderBy(
    F.desc("prediction")
)

top_recommendations = (
    scored
    .withColumn(
        "Recommendation_Rank",
        F.row_number().over(ranking)
    )
    .filter(F.col("Recommendation_Rank") <= 5)
)

# ==========================================
# 9. MAP NUMERIC IDS BACK TO ORIGINAL IDS
# ==========================================

mapped = (
    top_recommendations
    .join(
        customer_mapping,
        top_recommendations.user_id == customer_mapping.user_key
    )
    .join(
        product_mapping,
        top_recommendations.product_id == product_mapping.item_key
    )
    .join(
        products,
        product_mapping.product_key == products.Product_ID
    )
    .select(
        customer_mapping.customer_key.alias("Customer_ID"),
        products.Product_ID.alias("Product_ID"),
        products.Product_Name.alias("Product_Name"),
        products.Category.alias("Category"),
        products.Price.alias("Price"),
        products.Rating.alias("Rating"),
        top_recommendations.prediction.alias("Recommendation_Score"),
        top_recommendations.Recommendation_Rank.alias("Recommendation_Rank")
    )
)

# ==========================================
# 10. SAVE RECOMMENDATIONS
# ==========================================

mapped.write.format("delta").mode("overwrite").saveAsTable(
    "default.thames_product_recommendations_large"
)

print("THAMES ALS recommendations saved successfully!")

print(
    "Total recommendations:",
    spark.table(
        "default.thames_product_recommendations_large"
    ).count()
)

spark.table(
    "default.thames_product_recommendations_large"
).show(20, truncate=False)



Training interactions: 98971
ALS model trained successfully!
Unseen customer-product combinations: 4900529
THAMES ALS recommendations saved successfully!
Total recommendations: 49995
+-----------+----------+-----------------------------+-----------+-----+------+--------------------+-------------------+
|Customer_ID|Product_ID|Product_Name                 |Category   |Price|Rating|Recommendation_Score|Recommendation_Rank|
+-----------+----------+-----------------------------+-----------+-----+------+--------------------+-------------------+
|C00001     |P0145     |Electronics Accessories P0145|Electronics|12409|4.0   |0.92788124          |1                  |
|C00001     |P0133     |Footwear Shoes P0133         |Footwear   |9304 |4.7   |0.9154295           |2                  |
|C00001     |P0177     |Home Shoes P0177             |Home       |310  |4.8   |0.899255            |3                  |
|C00001     |P0076     |Electronics Fitness P0076    |Electronics|1228 |4.3   |0.8785582   

In [0]:
recommendations = spark.table(
    "default.thames_product_recommendations_large"
)

print("Total recommendations:", recommendations.count())

print(
    "Unique customers:",
    recommendations.select("Customer_ID").distinct().count()
)

print("Recommendations per customer:")
recommendations.groupBy("Customer_ID").count().groupBy("count").count().show()

recommendations.orderBy(
    "Customer_ID", "Recommendation_Rank"
).show(15, truncate=False)

Total recommendations: 49995
Unique customers: 9999
Recommendations per customer:
+-----+-----+
|count|count|
+-----+-----+
|    5| 9999|
+-----+-----+

+-----------+----------+-----------------------------+-----------+-----+------+--------------------+-------------------+
|Customer_ID|Product_ID|Product_Name                 |Category   |Price|Rating|Recommendation_Score|Recommendation_Rank|
+-----------+----------+-----------------------------+-----------+-----+------+--------------------+-------------------+
|C00001     |P0145     |Electronics Accessories P0145|Electronics|12409|4.0   |0.92788124          |1                  |
|C00001     |P0133     |Footwear Shoes P0133         |Footwear   |9304 |4.7   |0.9154295           |2                  |
|C00001     |P0177     |Home Shoes P0177             |Home       |310  |4.8   |0.899255            |3                  |
|C00001     |P0076     |Electronics Fitness P0076    |Electronics|1228 |4.3   |0.8785582           |4                  |


In [0]:

import mlflow
import mlflow.spark

from pyspark.sql import functions as F

# -----------------------------------
# 1. LOAD RECOMMENDATION RESULTS
# -----------------------------------

recommendations = spark.table(
    "default.thames_product_recommendations_large"
)

# Dataset statistics
training_count = training_data.count()
recommendation_count = recommendations.count()

customer_count = events.select("Customer_ID").distinct().count()
product_count = products.select("Product_ID").distinct().count()

recommended_customers = (
    recommendations.select("Customer_ID").distinct().count()
)

# -----------------------------------
# 2. START MLflow EXPERIMENT
# -----------------------------------

with mlflow.start_run(run_name="THAMES_ALS_Collaborative_Filtering"):

    # Model parameters
    mlflow.log_params({
        "algorithm": "ALS",
        "rank": 10,
        "max_iter": 10,
        "reg_param": 0.1,
        "implicit_prefs": True,
        "alpha": 20.0,
        "seed": 42
    })

    # Dataset and pipeline statistics
    mlflow.log_metrics({
        "training_interactions": float(training_count),
        "customer_count": float(customer_count),
        "product_count": float(product_count),
        "recommendation_count": float(recommendation_count),
        "recommended_customer_count": float(recommended_customers)
    })

    # Tags
    mlflow.set_tag("project", "THAMES")
    mlflow.set_tag("model_type", "Collaborative Filtering")
    mlflow.set_tag("framework", "Spark MLlib")
    mlflow.set_tag("recommendation_method", "ALS")

    # -----------------------------------
    # 3. LOG MODEL AND ID INDEXERS
    # -----------------------------------

    mlflow.spark.log_model(
        als_model,
        artifact_path="als_model",
        dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp"
    )

    mlflow.spark.log_model(
        customer_indexer_model,
        artifact_path="customer_indexer",
        dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp"
    )

    mlflow.spark.log_model(
        product_indexer_model,
        artifact_path="product_indexer",
        dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp"
    )

    run_id = mlflow.active_run().info.run_id

    print("THAMES ALS experiment logged successfully!")
    print("MLflow Run ID:", run_id)

print("Customers:", customer_count)
print("Products:", product_count)
print("Training interactions:", training_count)
print("Recommendations:", recommendation_count)


2026/11/05 16:20:52 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-836136b7-adc5-477f-8151-17/tmp8to3w30p/model, flavor: spark). Fall back to return ['pyspark==4.3.0']. Set logging level to DEBUG to see the full traceback. 
2026/11/05 16:20:52 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.12.0/ml/model/signatures.html for instructions on setting signature on models.
2026/11/05 16:21:02 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-836136b7-adc5-477f-8151-17/tmppuzs6f68/model, flavo

THAMES ALS experiment logged successfully!
MLflow Run ID: 09a7703b9582409dac6a52a41f9b1a5f
Customers: 9999
Products: 500
Training interactions: 98971
Recommendations: 49995


In [0]:
print("TRAINING DATA")
training_data.printSchema()

print("\nCUSTOMER MAPPING")
customer_mapping.printSchema()
display(customer_mapping.limit(5))

print("\nPRODUCT MAPPING")
product_mapping.printSchema()
display(product_mapping.limit(5))

print("\nALS MODEL")
print("Rank:", als_model.rank)

TRAINING DATA
root
 |-- user_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- rating: double (nullable = true)


CUSTOMER MAPPING
root
 |-- customer_key: string (nullable = true)
 |-- user_key: integer (nullable = true)



customer_key,user_key
C00520,519
C01843,1841
C00904,903
C00400,399
C06194,6192



PRODUCT MAPPING
root
 |-- product_key: string (nullable = true)
 |-- item_key: integer (nullable = true)



product_key,item_key
P0080,79
P0120,119
P0097,96
P0114,113
P0123,122



ALS MODEL
Rank: 10


In [0]:
from pyspark.sql import functions as F
from pyspark.ml.recommendation import ALS

# 1. Split interactions into training and testing data
eval_train, eval_test = training_data.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Evaluation training interactions:", eval_train.count())
print("Evaluation testing interactions:", eval_test.count())

# 2. Train a separate ALS evaluation model
eval_als = ALS(
    userCol="user_id",
    itemCol="product_id",
    ratingCol="rating",
    rank=10,
    maxIter=10,
    regParam=0.1,
    implicitPrefs=True,
    alpha=20.0,
    coldStartStrategy="drop",
    seed=42
)

eval_model = eval_als.fit(eval_train)

print("Evaluation ALS model trained successfully!")

# 3. Generate predictions for held-out interactions
predictions = eval_model.transform(eval_test)

print("Predictions generated:", predictions.count())

display(predictions.limit(10))

Evaluation training interactions: 79025
Evaluation testing interactions: 19946
Evaluation ALS model trained successfully!
Predictions generated: 19940


user_id,product_id,rating,prediction
0,196,1.0,0.6653206
0,254,1.0,0.7227382
1,199,1.0,-0.034480467
2,36,1.0,0.4955968
2,201,1.0,0.58664966
2,307,1.0,0.22557959
2,326,1.0,0.11887365
2,360,3.0,0.22251043
3,138,1.0,0.27379552
3,308,1.0,0.010836022


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Define relevant test interactions
# For implicit feedback, rating > 0 means the interaction is relevant.
relevant = (
    eval_test
    .filter(F.col("rating") > 0)
    .select("user_id", "product_id")
    .distinct()
)

# 2. Generate top 5 recommendations per user
ranking_window = Window.partitionBy("user_id").orderBy(
    F.desc("prediction"),
    F.asc("product_id")
)

top5 = (
    predictions
    .withColumn("rank", F.row_number().over(ranking_window))
    .filter(F.col("rank") <= 5)
)

# 3. Identify relevant recommendations
hits = (
    top5
    .join(
        relevant.withColumn("is_relevant", F.lit(1)),
        on=["user_id", "product_id"],
        how="left"
    )
    .fillna({"is_relevant": 0})
)

# 4. Calculate Precision@5 and Recall@5 per user
user_metrics = (
    hits
    .groupBy("user_id")
    .agg(
        F.sum("is_relevant").alias("hits"),
        F.count("*").alias("recommended_count")
    )
    .join(
        relevant.groupBy("user_id")
        .agg(F.count("*").alias("relevant_count")),
        on="user_id",
        how="inner"
    )
    .withColumn(
        "precision_at_5",
        F.col("hits") / F.lit(5.0)
    )
    .withColumn(
        "recall_at_5",
        F.col("hits") / F.col("relevant_count")
    )
)

# 5. Calculate NDCG@5
dcg = (
    hits
    .withColumn(
        "discount",
        F.lit(1.0) / (
            F.log2(F.col("rank").cast("double") + F.lit(1.0))
        )
    )
    .withColumn(
        "gain",
        F.col("is_relevant") * F.col("discount")
    )
    .groupBy("user_id")
    .agg(F.sum("gain").alias("dcg"))
)

# Ideal DCG for each user
ideal = (
    relevant
    .groupBy("user_id")
    .agg(F.count("*").alias("relevant_count"))
    .withColumn(
        "ideal_dcg",
        F.expr("""
            aggregate(
                sequence(1, least(relevant_count, 5)),
                cast(0.0 as double),
                (acc, x) -> acc + 1.0 / (log2(x + 1))
            )
        """)
    )
)

# 6. Combine metrics
final_metrics = (
    user_metrics
    .join(dcg, on="user_id", how="left")
    .join(ideal, on="user_id", how="inner")
    .withColumn(
        "ndcg_at_5",
        F.when(F.col("ideal_dcg") > 0,
               F.col("dcg") / F.col("ideal_dcg"))
         .otherwise(0.0)
    )
)

# 7. Calculate overall averages
evaluation_results = final_metrics.agg(
    F.avg("precision_at_5").alias("Precision@5"),
    F.avg("recall_at_5").alias("Recall@5"),
    F.avg("ndcg_at_5").alias("NDCG@5"),
    F.count("*").alias("Evaluated_Customers")
)

display(evaluation_results)


Precision@5,Recall@5,NDCG@5,Evaluated_Customers
0.45452655889144505,0.9954832471864781,1.0,8660


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Generate all customer-product candidate combinations
eval_users = eval_test.select("user_id").distinct()
all_products = training_data.select("product_id").distinct()

candidates = eval_users.crossJoin(all_products)

# 2. Remove products already seen during evaluation training
unseen_candidates = candidates.join(
    eval_train.select("user_id", "product_id").distinct(),
    on=["user_id", "product_id"],
    how="left_anti"
)

print("Unseen candidate pairs:", unseen_candidates.count())

# 3. Score candidates using the evaluation ALS model
candidate_scores = eval_model.transform(unseen_candidates)

candidate_scores = candidate_scores.filter(
    F.col("prediction").isNotNull()
)

# 4. Rank products for each customer
rank_window = Window.partitionBy("user_id").orderBy(
    F.desc("prediction"),
    F.asc("product_id")
)

top5_eval = (
    candidate_scores
    .withColumn("rank", F.row_number().over(rank_window))
    .filter(F.col("rank") <= 5)
)

# 5. Identify relevant held-out interactions
relevant_test = (
    eval_test
    .select("user_id", "product_id")
    .distinct()
)

hits = (
    top5_eval
    .join(
        relevant_test.withColumn("relevant", F.lit(1)),
        on=["user_id", "product_id"],
        how="left"
    )
    .fillna({"relevant": 0})
)

# 6. Calculate hits and Precision@5
user_hits = (
    hits.groupBy("user_id")
    .agg(
        F.sum("relevant").alias("hits"),
        F.count("*").alias("recommendation_count")
    )
)

user_relevant = (
    relevant_test.groupBy("user_id")
    .agg(F.count("*").alias("relevant_count"))
)

# 7. Calculate DCG
dcg = (
    hits
    .withColumn(
        "gain",
        F.col("relevant") /
        F.log2(F.col("rank").cast("double") + F.lit(1.0))
    )
    .groupBy("user_id")
    .agg(F.sum("gain").alias("dcg"))
)

# 8. Calculate final metrics
ideal = (
    user_relevant
    .withColumn(
        "ideal_dcg",
        F.expr("""
            aggregate(
                sequence(1, 5),
                cast(0.0 as double),
                (acc, x) -> acc + 1.0 / log2(x + 1)
            )
        """)
    )
)

final_evaluation = (
    user_hits
    .join(user_relevant, "user_id")
    .join(dcg, "user_id")
    .join(ideal.select("user_id", "ideal_dcg"), "user_id")
    .withColumn(
        "Precision_at_5",
        F.col("hits") / F.lit(5.0)
    )
    .withColumn(
        "Recall_at_5",
        F.col("hits") / F.col("relevant_count")
    )
    .withColumn(
        "NDCG_at_5",
        F.col("dcg") / F.col("ideal_dcg")
    )
)

# 9. Display overall evaluation
display(
    final_evaluation.agg(
        F.avg("Precision_at_5").alias("Precision@5"),
        F.avg("Recall_at_5").alias("Recall@5"),
        F.avg("NDCG_at_5").alias("NDCG@5"),
        F.count("*").alias("Evaluated_Customers")
    )
)

Unseen candidate pairs: 4262944


Precision@5,Recall@5,NDCG@5,Evaluated_Customers
0.004688221709006931,0.010503546684262621,0.004608332464418815,8660
